# Week_7_Lesson_Notebook_Simple_Prompt_Examples

**Description:** There are a variety of ways of accessing Gen AI models.  In this note book we will show you six very simple and free ones.<br>

Section 0 is environment set up.

Section 1 is about using the HuggingFace Transformer libraries and models in the repository.  Here we are using a new model from Hugging Face called SmolLM v3.  It stands out because it is relatively small at 3 billion paramters but has a 128K context window.  Let's look at [the model card](https://huggingface.co/HuggingFaceTB/SmolLM3-3B) from Hugging Face to get more background on just what distinguishes it from others.  Note it is optimized for common sense, language understanding, math, code, long context and logical reasoning.  Fortunetly we can leverage Hugging Face's quantization libraries and run the model on a T4 with some room to spare.

Section 2 uses another open sourced model called [Gemma](https://ai.google.dev/gemma) from Google.  These are lightweight models trained in the same way as Gemini but with a much smaller number of parameters.  

Section 3 is about using another year old model, called [Mistral](https://mistral.ai/), deployed as open source by a French start up in December  that is very performant despite it's small size.  This model is part of a current trend to improve efficiency and to squeeze more and more performance out of smaller and smaller models.  The regular 7B parameter model needs an A100 to run and uses up most of its GPU memory.  Fortunetly we can leverage Hugging Face's quantization libraries and run the model on a T4 with some room to spare.

Section 4 uses another open sourced model built for reasonong called [Qwen3](https://huggingface.co/Qwen/Qwen3-8B) from Alibaba.  These are dense models with a much smaller number of parameters that are distilled from a larger mixture of experts (MoE) model.  We can run these models on a T4 GPU in free Colab.

Section 5 uses another commercial service called [Cohere](https://dashboard.cohere.com/).  They offer a free playground and an API service to non-commercial users.  You will need to sign up and add your key as a secret in Colab in order to be able to use it.  You will need it for assignment 5.  We'll use their API to call their endpoints which include completion and embeddings.

Section 6 is about accessing [ChatGPT via a web interface](https://chat.openai.com/).  This allows you to experiment and to copy and paste but it does not allow you to access the model programiatically.  We'll talk about that later.

<a id = 'returnToTop'></a>

## Notebook Contents
  * 0. [Setup](#setup)
  * 1. [SmolLM3-3B](#smollm3)
  * 2. [Gemma 2](#gemma)
  * 3. [Mistral](#mistral7b-ift)
  * 4. [Qwen 3](#qwen3")
  * 5. [Cohere](#cohere)
  * 6. [ChatGPT](#chatgpt)
  
**To run this notebook** you should copy it to your Berkeley Google Drive or your personal Colab Plus Google account by uploading it into that Google Drive. From there you can open it as a Colab notebook and run it.  Note we were able to run it in the Free version of Colab but the RAM memory was maxed out.  It needs a T4 GPU to run Sections 1, 2, 3, and 4, but sections 5 and 6 access web services and therefore require way fewer resources and no GPU.

[Return to Top](#returnToTop)  
<a id = 'setup'></a>

# Setup

In [1]:
#let's make longer model output readable without horizontal scrolling
from pprint import pprint

In [2]:
%%capture
!pip install -q -U transformers


In [3]:
!pip install -q -U accelerate
!pip install -q -U bitsandbytes

In [4]:
import torch

device = "cuda:0" if torch.cuda.is_available() else "cpu"

This is the bits and bytes config file where we specify our quantization arguments.  You can read about it [here](https://huggingface.co/blog/4bit-transformers-bitsandbytes).

In [5]:
from transformers import BitsAndBytesConfig
import torch

quantization_config = BitsAndBytesConfig(
   load_in_4bit=True,
   bnb_4bit_quant_type="nf4",
   bnb_4bit_use_double_quant=True,
   bnb_4bit_compute_dtype=torch.bfloat16,
   llm_int8_enable_fp32_cpu_offload=True # Added to allow offloading to RAM if GPU is full
)

[Return to Top](#returnToTop)  
<a id = '#smollm3'></a>
## SmolLM3

We'll use a brand new model from Hugging Face called SmolLM v3.  It stands out because it is relatively small at 3 billion paramters but has a 128K context window.  Let's look at [the model card](https://huggingface.co/HuggingFaceTB/SmolLM3-3B) from Hugging Face to get more background on just what distinguishes it from others.  Note it is optimized for common sense, language understanding, math, code, long context and logical reasoning.  They provide [an excellent and comprehensive description of how it was trained](https://huggingface.co/blog/smollm3).  We talked about quantization in week 5.

In [6]:

from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

torch.random.manual_seed(0)

We're going to quantize our model which will shrink its memory footprint without reducing its performance in any significant way.  We'll discuss quantization in a later session.

In [7]:
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,

)

The prompt structure includes two components.  You can add the /no_think command to the system component to turn off "thinking." If you comment it out then it reverts to its default bheavior which is to do long thinking.  Make sure that you change the value of max_new_tokens to insure you can emit **BOTH** the thinking AND the answer tokens.

In [8]:

messages = [
    {"role": "system", "content": "/no_think"},    #turn off reasoning- "/think" or nothing to turn on
    {"role": "user", "content": "write a three sentence description for the following product: Cuisinart Airfryer, 6-Qt Basket Air Fryer Oven that Roasts, Bakes, Broils & Air Frys Quick & Easy Meals - Digital Display with 5 Presets, Non Stick & Dishwasher Safe, AIR-200"},
]

In [9]:
# Use a pipeline as a high-level helper
from transformers import pipeline

model_id="HuggingFaceTB/SmolLM3-3B"
#model_id="allenai/Olmo-3-7B-Instruct"

# This will load the FULL 3B paramater model with quantization ~ 3.5 of 15.0GB and 3 minutes to respond
smol_pipe = pipeline("text-generation", model=model_id, model_kwargs={"dtype": torch.bfloat16, "quantization_config": quantization_config},
    device_map="auto",)

outputs = smol_pipe(messages, max_new_tokens=1024,)       #shorter token length here will hurt reasoning but we're going /no_think


pprint(outputs[0]["generated_text"][-1], compact=True)

config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/26.9k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/5.60k [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips sp

{'content': 'The Cuisinart Airfryer, 6-Qt Basket Air Fryer Oven is a versatile '
            'kitchen appliance that roasts, bakes, broils, and air fries quick '
            'and easy meals. With a digital display and 5 presets, it offers '
            'easy meal preparation with a variety of cooking options. Its '
            'non-stick surface and dishwasher-safe design make it both '
            'convenient and practical for everyday use.',
 'role': 'assistant'}


## Short vs. Long Thought Models

We'll be using SmolLM3, a hybrid model designed to give either thoughtful or quick answers.  Sometimes, depending on your problem, such a short thought model is what's best for you.  Other times, like if doing math, logic, or puzzles then a longer thought reasoning model is most approrpaite.  You can change this via a hyperparameter - `think` or `no_think` -  and the default is the longer thought mode.  After you've run all our prompts with the short thought model you can comment the reasoning model line with `no_think` and try the long reasoning model to see how it performs.

What is the meaning of the string - `HuggingFaceTB/SmolLM3-3B`.  The `HuggingFaceTB` portion means it comes from Hugging Face. `SmolLM3` is the name of the model.  `3B` refers to the variant of the model usually indicating the number of parameters. Finally, there is another model called `HuggingFaceTB/SmolLM3-3B-Base` means this model has only been pre-trained and not post-trained so it should not be good at following our instructions.

We'll continue to run the model fully postrained model.  We'll construct our prompt which we'll put in the messages list.  Note that the model is trained to do some dialog.  We can toggle back and forth between the 'user' and 'assistant' roles.  We can also just feed in the initial 'user' field if we just want one prompt.

Since this model is pre-trained for math and logical reasoning, let's try it with a simple equation.

In [10]:
#This shows the turn-taking conversational approach. The first user and assistant pair represents the first turn in the conversation.
#We'll typically just use one 'user' input.
messages = [
    {"role": "system", "content": "/no_think"},    #turn off reasoning- "/think" or nothing to turn on
    {"role": "user", "content": "Can you provide ways to eat combinations of bananas and dragonfruits?"},
    {"role": "assistant", "content": "Sure! Here are some ways to eat bananas and dragonfruits together: 1. Banana and dragonfruit smoothie: Blend bananas and dragonfruits together with some milk and honey. 2. Banana and dragonfruit salad: Mix sliced bananas and dragonfruits together with some lemon juice and honey."},
    {"role": "user", "content": "What about solving an 2x + 3 = 7 equation?"},
]

outputs = smol_pipe(messages, max_new_tokens=2048,)       #shorter token length here will hurt reasoning

pprint(outputs[0]['generated_text'],compact=True)


[transformers] Both `max_new_tokens` (=2048) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'content': '/no_think', 'role': 'system'},
 {'content': 'Can you provide ways to eat combinations of bananas and '
             'dragonfruits?',
  'role': 'user'},
 {'content': 'Sure! Here are some ways to eat bananas and dragonfruits '
             'together: 1. Banana and dragonfruit smoothie: Blend bananas and '
             'dragonfruits together with some milk and honey. 2. Banana and '
             'dragonfruit salad: Mix sliced bananas and dragonfruits together '
             'with some lemon juice and honey.',
  'role': 'assistant'},
 {'content': 'What about solving an 2x + 3 = 7 equation?', 'role': 'user'},
 {'content': 'To solve the equation 2x + 3 = 7, you can follow these steps:\n'
             '\n'
             'Step 1: Subtract 3 from both sides of the equation to isolate '
             'the term with the variable (2x) on one side.\n'
             '2x + 3 - 3 = 7 - 3\n'
             '2x = 4\n'
             '\n'
             'Step 2: Divide both sides of the equation by 

What if we ask the model to write a review of a product based solely on naming the product but without other information.  Let's also make the voice a variable.

In [11]:
#Try some different tasks/prompts
myvoice = "millenial parent"     #change this to get interesting results
myprompt = f"Write a very positive three sentence review in the voice of a {myvoice} for a Cuisinart 6-Qt Airfryer"

messages = [
    {"role": "system", "content": "/no_think"},    #turn off reasoning- "/think" or nothing to turn on
        {"role": "user", "content": myprompt}
]

#encodeds = tokenizer.apply_chat_template(messages, return_tensors="pt")

outputs = smol_pipe(messages, max_new_tokens=1024,)       #shorter token length here will hurt reasoning


pprint(outputs[0]["generated_text"][-1], compact=True)

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'content': 'What a game-changer for our family! The Cuisinart 6-Qt Airfryer '
            'has transformed our cooking routine, making it so much easier to '
            'enjoy healthy, crispy meals without the mess and fuss. My kids '
            'love how fun and interactive it is, and now we can enjoy a '
            'variety of tasty treats that are actually good for us too!',
 'role': 'assistant'}


Now we'll ask the model to generate a review of the product using the product description we just generated.  We'll also ask that the model generate the review in the voice of a baby boomer.

In [12]:
myprompt = (
    "Write a very positive three sentence review in the voice of a boomer based on the following product description: "
    "The Cuisinart Airfryer, model number AIR-200, is a 6-Qt capacity basket air fryer oven that allows for roasting, "
    "baking, broiling, and air frying to prepare quick and easy meals. Its 'digital display includes 5 preset functions, "
    "making cooking convenient and hassle-free. Featuring a non-stick interior and dishwasher safe parts, it ensures effortless "
    "cleanup."
)

messages = [
    {"role": "system", "content": "/no_think"},    #turn off reasoning- "/think" or nothing to turn on
        {"role": "user", "content": myprompt}
]

outputs = smol_pipe(messages, max_new_tokens=2048,)       #shorter token length here will hurt reasoning


pprint(outputs[0]["generated_text"][-1], compact=True)


[transformers] Both `max_new_tokens` (=2048) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'content': 'What a delightful find! The Cuisinart Airfryer is a real '
            'game-changer for my cooking routine. With its 6-Qt capacity and 5 '
            'convenient preset functions, I can whip up delicious meals in no '
            "time, and the non-stick interior makes cleanup a breeze. I'm so "
            "glad I found it, it's perfect for my boomer lifestyle!",
 'role': 'assistant'}


Now we'll ask the model to rewrite its previous review in the voice of a Gen Z gamer.  We can use this kind of functionality to generate synthetic data that can augment our data set if we have small amounts of data or if we have a class imbalance.

In [13]:
myprompt = (
    "Write a very positive three sentence review in the voice of a Gen Z gamer based on the following product description: "
    "The Cuisinart Airfryer, model number AIR-200, is a 6-Qt capacity basket air fryer oven that allows for roasting, "
    "baking, broiling, and air frying to prepare quick and easy meals. Its 'digital display includes 5 preset functions, "
    "making cooking convenient and hassle-free. Featuring a non-stick interior and dishwasher safe parts, it ensures effortless "
    "cleanup."
)

messages = [
    {"role": "system", "content": "/no_think"},    #turn off reasoning- "/think" or nothing to turn on
        {"role": "user", "content": myprompt}
]

outputs = smol_pipe(messages, max_new_tokens=1024,)       #shorter token length here will hurt reasoning


pprint(outputs[0]["generated_text"][-1], compact=True)

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'content': "As a Gen Z gamer, I just couldn't resist picking up the Cuisinart "
            "Airfryer, model AIR-200. With its 6-Qt capacity basket, it's the "
            'perfect size for quick and easy meals, perfect for after a long '
            'gaming session. The digital display with 5 preset functions is a '
            'lifesaver, making cooking a breeze.',
 'role': 'assistant'}


In [ ]:
#Simple cell to experiment
#Try some different tasks/prompts
myprompt = (
    "something, anything?",
    ""
)

messages = [
    {"role": "system", "content": "/no_think"},    #turn off reasoning- "/think" or nothing to turn on
        {"role": "user", "content": myprompt}
]

outputs = smol_pipe(messages, max_new_tokens=1024,)       #shorter token length here will hurt reasoning


pprint(outputs[0]["generated_text"][-1], compact=True)

{'content': "I'm here to help you with your queries and analysis. Feel free to "
            'ask me any questions or provide data that you need help with, '
            "whether it's related to mathematics, programming, data analysis, "
            'or any other topic. Just paste the text or the data you want me '
            "to analyze, and I'll do my best to assist you.",
 'role': 'assistant'}


Because of the extensive pre-training of [the model](https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2), it has many different capabilities.  Let's see if it can translate Chinese into English.

In [14]:
myprompt = (
    "Translate the following Chinese into English: boys, 我想告诉你，这个 "
 "CuisinartAirfryer，模型号是AIR-200，真是一件事! 六quirelit的容量,你问？绝对足够丰盛家人晚餐。接下来，烤、烘、�� "
 "broil和空炐烤全在一个机器上，就像拥有整个厨房一样。但是等下， Digital display你知道什么？五种预设功能？就像拥有personal "
 "chef一样，使cooking轻松无难度。最後，非沾锅内部和洗装盘，清理真是可以比作�arts simple as "
 "pie。简直神赞，这个Airfryer真是棒棒哇!"

)

messages = [
    {"role": "system", "content": "/no_think"},    #turn off reasoning- "/think" or nothing to turn on
        {"role": "user", "content": myprompt}
]

outputs = smol_pipe(messages, max_new_tokens=1024,)       #shorter token length here will hurt reasoning


pprint(outputs[0]["generated_text"][-1], compact=True)

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'content': 'Boys, I want to tell you something. This Cuisinart Airfryer, '
            'model number AIR-200, is just incredible! The capacity, you ask? '
            'Absolutely enough to feed a full family dinner. Next, you can '
            'bake, roast, broil, and air fry all in one machine, just like '
            'having an entire kitchen. Wait, though, do you know what the '
            'digital display is? Five preset functions? Just like having a '
            'personal chef, making cooking a breeze. Finally, the inner '
            'non-stick pan and the dishwasher-safe bowl, cleaning up is as '
            "easy as pie. It's absolutely amazing, this Airfryer is fantastic!",
 'role': 'assistant'}


Let's run some of the same prompts that we ran above to see how well this model performs.  Note that it takes a lot longer to generate answers because this model has 8 billion rather than 3 billion parameters.  The next cell takes about 2 minutes to complete.

How well do the outputs from Llama3.1 compare with the outputs from Gemma 2?  How can we measure their performance? How can we compare the two models quantitatively?

[Return to Top](#returnToTop)  
<a id = 'gemma'></a>

## Gemma 2


[Gemma 2](https://huggingface.co/google/gemma-2-9b-it) is a model produced by Google.  We'll use the 9B model that's been instuction fine-tuned.  We'll use quantization to be able to run it in free Colab and its base GPU.

Why are we looking at a second model? Because the behavior of these models is idiosyncratic and it is important to try multiple models to identify these differences.

In [15]:
# pip install bitsandbytes accelerate
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_id = "google/gemma-2-9b-it"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=quantization_config, device_map="auto")

input_text = "Write me a poem about Machine Learning."
input_ids = tokenizer(input_text, return_tensors="pt").to("cuda")

outputs = model.generate(**input_ids)
pprint(tokenizer.decode(outputs[0]), compact=True)

config.json:   0%|          | 0.00/857 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/39.1k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1616: UserWarning: Using the model-agnostic default `max_length` (=29) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


('<bos>Write me a poem about Machine Learning.\n'
 '\n'
 'A mind of code, a web of thought,\n'
 'Machine Learning, lessons taught.\n'
 'From')


That's a really short poem.  Too short and ending on a comma is suspicious.  Why did this happen?

Hint: what are the defaults?

In [16]:
prompt_text = "Write me a very postive review of a Cuisinart 6 Qt Air Fryer"
input_ids = tokenizer(prompt_text, return_tensors="pt").to("cuda")

outputs = model.generate(**input_ids, max_new_tokens=512)
pprint(tokenizer.decode(outputs[0]), compact=True)

('<bos>Write me a very postive review of a Cuisinart 6 Qt Air Fryer\n'
 '\n'
 '**Crispy Perfection, Every Time!**\n'
 '\n'
 "I've been using my Cuisinart 6 Qt Air Fryer for a few weeks now, and I'm "
 'absolutely in love! This thing is a game-changer in my kitchen. \n'
 '\n'
 'First off, the size is perfect. 6 quarts is plenty of space for my family of '
 'four, and I can easily fit a whole chicken or a large batch of fries in '
 'there. The air fryer cooks everything evenly and quickly, and the results '
 'are always crispy and delicious. \n'
 '\n'
 "I've tried everything from chicken wings to french fries to vegetables, and "
 'everything has come out amazing. The air fryer makes it so easy to cook '
 'healthy meals without sacrificing flavor. \n'
 '\n'
 'The Cuisinart air fryer is also incredibly easy to use. The controls are '
 'simple and intuitive, and the digital display makes it easy to see the time '
 'and temperature. Cleaning is a breeze too, thanks to the dishwasher-safe '


In [17]:
prompt_text = "Write a very positive three sentence review in the voice of a boomer based on the following product description: The Cuisinart Airfryer, model number AIR-200, is a 6-Qt capacity basket air fryer oven that allows for roasting, baking, broiling, and air frying to prepare quick and easy meals. Its 'digital display includes 5 preset functions, making cooking convenient and hassle-free. Featuring a non-stick interior and dishwasher safe parts, it ensures effortless cleanup."

input_ids = tokenizer(prompt_text, return_tensors="pt").to("cuda")

outputs = model.generate(**input_ids, max_new_tokens=512)
pprint(tokenizer.decode(outputs[0]), compact=True)

('<bos>Write a very positive three sentence review in the voice of a boomer '
 'based on the following product description: The Cuisinart Airfryer, model '
 'number AIR-200, is a 6-Qt capacity basket air fryer oven that allows for '
 'roasting, baking, broiling, and air frying to prepare quick and easy meals. '
 "Its 'digital display includes 5 preset functions, making cooking convenient "
 'and hassle-free. Featuring a non-stick interior and dishwasher safe parts, '
 'it ensures effortless cleanup.\n'
 '\n'
 "This Cuisinart Air Fryer is a real game changer!  It's so easy to use, even "
 'my grandkids could figure it out, and the food comes out crispy and '
 'delicious every time.  I love that it can do so much more than just fry, '
 "like roast and bake, so I'm using it all the time now.\n"
 '\n'
 '\n'
 '<end_of_turn><eos>')


In [18]:
prompt_text = "Q: A juggler can juggle 16 balls.  Half of the balls are golf balls and half of the golf balls are blue.  How many blue golf balls are there? A: Let's think step by step. "
input_ids = tokenizer(prompt_text, return_tensors="pt").to("cuda")

outputs = model.generate(**input_ids, max_new_tokens=200)
pprint(tokenizer.decode(outputs[0]), compact=True)

('<bos>Q: A juggler can juggle 16 balls.  Half of the balls are golf balls and '
 'half of the golf balls are blue.  How many blue golf balls are there? A: '
 "Let's think step by step. 1. Find the number of golf balls: 16 balls / 2 = 8 "
 'golf balls 2. Find the number of blue golf balls: 8 golf balls / 2 = 4 blue '
 'golf balls Answer: There are **4** blue golf balls.<end_of_turn>\n'
 '<eos>')


[Return to Top](#returnToTop)  
<a id = 'mistral-ift'></a>

## Mistral 7B -

[Mistral 7B](https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2) is a small but highly performant model. It is also possible to use it commercially. The model has been instruction fine-tuned by Mistral.ai so it should be able to follow ou prompts and return good on point output.  We'll also use a quantixzed version (down to 4 bits) so we know it can load in our small GPU.  

If you've already run the Llama and Gemma models, you need to shutdown and disconnect your session.  Then reconnect, run setup at the top of the notebook to load the libaries necessary for it to work.  Then return here and you can try the Mistral model.

This model has been trained to work with dialog, meaning instances where we have multiple utterance and response pairs to create the context so the model can reply. For these examples we'll populate the context with only our prompt and not have any back and forth.

First we'll ask the model to generate a product desciption based on the content of the product title.

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer

In [7]:
myprompt = (
    "write a three sentence description for the following product: Cuisinart Airfryer, 6-Qt Basket Air Fryer Oven that Roasts, "
    "Bakes, Broils & Air Frys Quick & Easy Meals - Digital Display with 5 Presets, Non Stick & Dishwasher Safe, AIR-200"
    )

In [8]:
model_id = "mistralai/Mistral-7B-Instruct-v0.3"
#Note: It can take up to 3 minutes to download this model

from transformers import AutoModelForCausalLM, AutoTokenizer
from pprint import pprint
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

# Ensure quantization_config is defined (assumes BitsAndBytesConfig is imported/available from earlier cells)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=quantization_config, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_id)

messages = [
    {"role": "user", "content": myprompt}
]

# apply_chat_template returns a BatchEncoding or list of IDs
encodeds = tokenizer.apply_chat_template(messages, return_tensors="pt")

# Move all components of the encoded input to the device
model_inputs = encodeds.to(device)

# Fixed: Use **model_inputs to unpack the dictionary into keyword arguments
generated_ids = model.generate(**model_inputs, max_new_tokens=1000, do_sample=True)
decoded = tokenizer.batch_decode(generated_ids)
pprint(decoded[0], compact=True)

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


...

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


('<s>[INST] write a three sentence description for the following product: '
 'Cuisinart Airfryer, 6-Qt Basket Air Fryer Oven that Roasts, Bakes, Broils & '
 'Air Frys Quick & Easy Meals - Digital Display with 5 Presets, Non Stick & '
 'Dishwasher Safe, AIR-200[/INST] The Cuisinart Airfryer, 6-Qt Basket Air '
 'Fryer Oven is a versatile kitchen appliance that offers roasting, baking, '
 'broiling, and air frying capabilities for quick and easy meal preparation. '
 'It features a digital display with 5 presets for convenience, and its '
 'non-stick, dishwasher-safe components ensure easy cleanup. With this device, '
 'you can enjoy delicious, healthy meals at home.</s>')


In [9]:
print(model.quantization_method)

QuantizationMethod.BITS_AND_BYTES


What if we ask the model to write a review of a product based solely on naming the product but without other information.  Let's also make the voice a variable.

In [12]:
#Try some different tasks/prompts
myvoice = "millenial parent"
myprompt = f"Write a very positive three sentence review in the voice of a {myvoice} for a Cuisinart 6-Qt Airfryer"

messages = [
        {"role": "user", "content": myprompt}
]

encodeds = tokenizer.apply_chat_template(messages, return_tensors="pt")

model_inputs = encodeds.to(device)
#model.to(device)

# Fixed: Added ** to unpack model_inputs into keyword arguments
generated_ids = model.generate(**model_inputs, max_new_tokens=1000, do_sample=True)
decoded = tokenizer.batch_decode(generated_ids)
pprint(decoded[0], compact=True)

[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


('<s>[INST] Write a very positive three sentence review in the voice of a '
 'millenial parent for a Cuisinart 6-Qt Airfryer[/INST] As a busy millennial '
 "parent, I can't help but rave about the Cuisinart 6-Qt Airfryer! This "
 'innovative kitchen appliance has become a staple in our home. Not only does '
 'it allow me to whip up delicious, crispy meals in a flash, but it also helps '
 'us maintain a healthier lifestyle by reducing the amount of oil used in our '
 'cooking. The sleek design and easy-to-use controls make it a pleasure to '
 'use, and the clean-up process is a breeze. This Airfryer is truly a '
 'game-changer for our family meal times!</s>')


[Return to Top](#returnToTop)  
<a id = '#qwen3'></a>
## Qwen 3

Qwen 3 is the latest open source model from Alibaba's Qwen group. We'll used it extensively in this class. Check out [the model card](https://huggingface.co/Qwen/Qwen3-8B) for further details. It makes it easy to toggle between "thinking" and "non-thinking" modes.  This allows us to easily see if the added thinking helps or hurts the perfrmence of the model.  We're using the 8 billion parameter version but quantized so it has a much smaller memory footprint.  We talked about quantization in week 5.

This way loads the model using the AutoModel abstraction from Hugging Face.  You have access to several ways of interacting with a model.  One of those is through the AutoModel and the AutoTokenizer classes.  The other is via the Pipeline class.  Let's look very briefly at the AutoModel approach.

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3-8B" #runs barely in T4 so you must quantize

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto",
    quantization_config=quantization_config
)

# prepare the model input
prompt = "Give me a short introduction to large language models."
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False # Switches between thinking and non-thinking modes. Default is True.
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

# parsing thinking content - separate thoughts from final answer
try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

pprint(("thinking content:", thinking_content), compact=True)
pprint(("content:", content), compact=True)


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


('thinking content:', '')
('content:',
 'Large language models (LLMs) are advanced artificial intelligence systems '
 'designed to understand and generate human-like text. These models are '
 'trained on vast amounts of text data from the internet, books, and other '
 'sources, allowing them to recognize patterns, grasp context, and produce '
 'coherent responses to a wide range of queries. LLMs can perform tasks such '
 'as answering questions, writing stories, coding, and even engaging in '
 'conversations. Their ability to process and generate natural language has '
 'made them a powerful tool in various fields, from customer service to '
 'content creation and research.')


Let's run some of the same prompts that we ran above to see how well this model performs.  Note that it takes a lot longer to generate answers because this model "thinks" before it answers.  The next cell can take about 2 minutes to complete.

How well do the outputs from Qwen 3 compare with the outputs from the previous models?  How can we measure their performance? How can we compare the two models' outputs quantitatively?

In [7]:
# prepare the model input
messages = [
    {"role": "system", "content": "You are a science communicator who makes technology accessible to everyone!"},
    {"role": "user", "content": "Please write a five sentence explanation of how LLMs do knowledge representation."},
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True # Switches between thinking and non-thinking modes. Default is True.
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

# parsing thinking content
try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

pprint(("thinking content:", thinking_content), compact=True)
pprint(("content:", content), compact=True)

('thinking content:',
 '<think>\n'
 'Okay, the user wants a five-sentence explanation of how LLMs do knowledge '
 'representation. Let me start by recalling what I know about LLMs. They '
 'process text by converting words into vectors, right? So maybe I should '
 'mention embeddings. But how do they represent knowledge beyond just words?\n'
 '\n'
 'Hmm, LLMs use layers of neural networks to capture patterns. Each layer '
 'might abstract information, like from individual words to phrases, then to '
 'sentences. But how does that translate to knowledge? Maybe through context '
 'and relationships between words. They learn associations, like synonyms or '
 'related concepts.\n'
 '\n'
 'Wait, the user might be looking for how the model stores and retrieves '
 "information. So, it's not just about word vectors but the entire context. "
 "The model's parameters hold learned knowledge, which is distributed across "
 'the weights. But explaining that might be too technical. I should simplify

A second way of interacting with these models is to use the Pipeline class.

In [8]:
# Use a pipeline as a high-level helper
from transformers import pipeline

messages = [
    {"role": "user", "content": "How many r's in raspberry?"},
]

# uncomment the following to run with quantization ~ 7.0 of 15.0GB and 3 minutes to respond
pipe = pipeline("text-generation", model="Qwen/Qwen3-8B", model_kwargs={"dtype": torch.bfloat16, "quantization_config": quantization_config, },
    device_map="auto",)

outputs = pipe(messages, max_new_tokens=2048,)


pprint(outputs[0]["generated_text"][-1], compact=True)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=2048) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


{'content': '<think>\n'
            "Okay, the user is asking how many times the letter 'r' appears in "
            'the word "raspberry". Let me start by writing out the word: '
            'R-A-S-P-B-E-R-R-Y.\n'
            '\n'
            'Wait, let me check each letter one by one. The first letter is '
            "'R', that's one. Then 'A', 'S', 'P', 'B', 'E', then the next "
            'letters. Let me go step by step. \n'
            '\n'
            'First letter: R (1). Then A, S, P, B, E. Next, R again? Wait, '
            "after E, the next letters are R, R, Y. So after E, it's R, then "
            "another R, then Y. So that's two more R's. Let me count again. \n"
            '\n'
            'Breaking down "raspberry" into individual letters: R, A, S, P, B, '
            "E, R, R, Y. So that's R at the beginning, then later two R's. So "
            "total of three R's. Wait, but let me make sure I didn't miss any. "
            "Let me spell it again: R-A-S-P-B-E-R-R-

By default, Qwen 3 will think.  However you can turn it off by simply adding `/no_think` at the end of your prompt.

In [9]:
#let's try a second example
messages = [
    {"role": "user", "content": "What is the ratio of the circumfrence of the world to the circumference of the sun? /no_think"},
]

outputs = pipe(messages, max_new_tokens=1024)


pprint(outputs[0]["generated_text"][-1], compact=True)

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'content': '<think>\n'
            '\n'
            '</think>\n'
            '\n'
            'To find the ratio of the **circumference of the Earth** to the '
            '**circumference of the Sun**, we can follow these steps:\n'
            '\n'
            '---\n'
            '\n'
            '### **Step 1: Find the Circumference of the Earth**\n'
            '\n'
            'The Earth is approximately a sphere with an **average radius** '
            'of:\n'
            '\n'
            '$$\n'
            'r_{\\text{Earth}} \\approx 6,371 \\text{ km}\n'
            '$$\n'
            '\n'
            'The formula for the circumference of a circle is:\n'
            '\n'
            '$$\n'
            'C = 2\\pi r\n'
            '$$\n'
            '\n'
            'So, the **circumference of the Earth** is:\n'
            '\n'
            '$$\n'
            'C_{\\text{Earth}} = 2 \\pi \\times 6,371 \\approx 40,075 \\text{ '
            'km}\n'
            '$$\n'
            '\n'

[Return to Top](#returnToTop)  
<a id = 'cohere'></a>

## Cohere


[Cohere](cohere.com) is a company with commercial LLMs and endpoints you can access via an API call. They let you try before you buy so they offer a free account to non-commercial entities which will give you an API key.  You should open a free account and get an API key.  

**DO NOT PUT THE API KEY DIRECTLY INTO YOUR NOTEBOOK**

Make sure you either use Colab secrets or a .env file so the key is hidden from view.

We'll use the Cohere embeddings endpoint in a future assignment so now is the time to get your account open.

In [10]:
#access the hidden API key
from google.colab import userdata
COHERE_API_KEY = userdata.get('COHERE_API_KEY')

In [11]:
import pprint

In [12]:
!pip install -q cohere

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 352.0/352.0 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 94.2 MB/s eta 0:00:00


This prompt includes a paragraph from [Wikipedia](https://en.wikipedia.org/wiki/Grammar) about grammar.  The paragraph contains a number of places and dates.  We can use the model to extract a number of the entities and return it to us in a JSON record.  It can be a powerful way of extracting value from noisy data.

In [13]:
myprompt = (
    "Please identify all of the places and dates in the following paragraph "
    "and return them in a JSON record with fields for date and loc: "
    "Belonging to the trivium of the seven liberal arts, grammar was taught "
    "as a core discipline throughout the Middle Ages, following the influence "
    "of authors from Late Antiquity, such as Priscian. Treatment of vernaculars "
    "began gradually during the High Middle Ages, with isolated works such as "
    "the First Grammatical Treatise, but became influential only in the Renaissance "
    "and Baroque periods. In 1486, Antonio de Nebrija published Las introduciones "
    "Latinas contrapuesto el romance al Latin, and the first Spanish grammar, "
    "Gramática de la lengua castellana, in 1492. During the 16th-century Italian "
    "Renaissance, the Questione della lingua was the discussion on the status and "
    "ideal form of the Italian language, initiated by Dante's de vulgari eloquentia "
    "(Pietro Bembo, Prose della volgar lingua Venice 1525). The first grammar of "
    "Slovene was written in 1583 by Adam Bohorič, and Grammatica Germanicae Linguae, "
    "the first grammar of German, was published in 1578."
)


In [19]:
import cohere
co = cohere.Client(COHERE_API_KEY)

# Using a specific versioned model as the unversioned aliases were retired.
response = co.chat(
  message=myprompt,
  model='command-r-plus-08-2024',  #try command-a-03-2025
  temperature=0.5,
  p=0.96
)

import pprint
pprint.pprint(f'Prediction: {response.text}', compact=True)

('Prediction: [\n'
 '    {\n'
 '        "date": "1486",\n'
 '        "loc": "Las introduciones Latinas contrapuesto el romance al Latin"\n'
 '    },\n'
 '    {\n'
 '        "date": "1492",\n'
 '        "loc": "Gramática de la lengua castellana"\n'
 '    },\n'
 '    {\n'
 '        "date": "1525",\n'
 '        "loc": "Venice"\n'
 '    },\n'
 '    {\n'
 '        "date": "1578",\n'
 '        "loc": "Grammatica Germanicae Linguae"\n'
 '    },\n'
 '    {\n'
 '        "date": "1583",\n'
 '        "loc": "Slovene"\n'
 '    }\n'
 ']')


[Return to Top](#returnToTop)  
<a id = 'chatgpt'></a>
## 3. ChatGPT

You can access a [free version of ChatGPT here](https://chat.openai.com/).  You will need to create an account (unless you already have one) using your personal and **NOT your @berkeley.edu account**.  You want to use ChatGPT 4o-mini and not anything higher.  This model can only be accessed via a web browser.  Later we will create pay as you go accounts that will allow you call these models from a notebook using your unique key.  We'll do it in a way that minimizes your expenses.

Let's access ChatGPT and try a couple of super simple prompts to see how well it performs.  It can be a very handy tool. (If you prefer you can repeat the prompts we used with Mistral)

Prompt 1 - ```write a blurb for the following book: Lee McIntyre, “On Disinformation: How to Fight for Truth and Protect Democracy” (MIT 2023)```

Prompt 2 - ```write a three sentence review of the book described in the following blurb: <insert generated blurb>```

Prompt 3 - ```What is the sentiment expressed in this review:```

Prompt 4 - ```Write a negative and pedantic three sentence review of the book described in the following blurb:```
  

How did those perform?  

Is the performance between the three models equivalent or are there noticeable differences.